In [1]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 66.9 MB/s  0:00:01


In [4]:
import numpy as np
import pandas as pd
import joblib
from catboost import CatBoostRegressor
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline

In [16]:
df = pd.read_excel('/content/cases_train2 (15).xlsx')
df['Оценка'] = df['Оценка'].astype('float64')
df_test_cases = pd.read_excel('/content/Копия Test1.xlsx')
df_test_cases['Оценка'] = df_test_cases['Оценка'].astype('float64')

In [17]:
texts_train = df["Решение кейса"].astype(str).str.replace(r'<br\s*/?>', ' ', regex=True)
texts_test = df_test_cases["Решение кейса"].astype(str).str.replace(r'<br\s*/?>', ' ', regex=True)

In [18]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [19]:
# Модель для оценивания критерия Анализ ЦА
y_train_audience = df["ЦА"].astype(float).values
y_test_audience = df_test_cases["ЦА"].astype(float).values
model_audience = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True, max_features=20000)),
    ("svd", TruncatedSVD(n_components=100, random_state=42)),
    ("catboost", CatBoostRegressor(iterations=300, depth=3, learning_rate=0.05,
                                   l2_leaf_reg=10, rsm=0.5, verbose=0, random_seed=42)),
])
pred_cv_audience = np.clip(cross_val_predict(model_audience, texts_train, y_train_audience, cv=kf), 1, 5)

In [20]:
print("MAE Оценка ЦА Val", mean_absolute_error(y_train_audience, pred_cv_audience))
model_audience.fit(texts_train, y_train_audience)
pred_test_audience = np.clip(model_audience.predict(texts_test), 1, 5)
print("MAE Оценка ЦА Test", mean_absolute_error(y_test_audience, pred_test_audience))

MAE Оценка ЦА Val 0.5785380139851302
MAE Оценка ЦА Test 0.6469637756035947


In [21]:
#Модель для оценивания критерия Проработка решения
y_train_solution = df["Проработка решения"].astype(float).values
y_test_solution = df_test_cases["Проработка решения"].astype(float).values
model_solution = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True, max_features=20000)),
    ("svd", TruncatedSVD(n_components=100, random_state=42)),
    ("catboost", CatBoostRegressor(iterations=300, depth=3, learning_rate=0.05,
                                   l2_leaf_reg=10, rsm=0.5, verbose=0, random_seed=42)),
])
pred_cv_solution = np.clip(cross_val_predict(model_solution, texts_train, y_train_solution, cv=kf), 1, 5)
print("MAE Val Проработка решения", mean_absolute_error(y_train_solution, pred_cv_solution))
model_solution.fit(texts_train, y_train_solution)
pred_test_solution = np.clip(model_solution.predict(texts_test), 1, 5)
print("MAE Test Проработка решения", mean_absolute_error(y_test_solution, pred_test_solution))


MAE Val Проработка решения 0.6145789862273776
MAE Test Проработка решения 0.6996936327494631


In [22]:
#Модель для оценивания критерия Финансовая модель и метрики
y_train_finance = df["Финансовая модель и метрики"].astype(float).values
y_test_finance = df_test_cases["Финансовая модель и метрики"].astype(float).values
model_finance = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True, max_features=20000)),
    ("svd", TruncatedSVD(n_components=100, random_state=42)),
    ("catboost", CatBoostRegressor(iterations=300, depth=3, learning_rate=0.05,
                                   l2_leaf_reg=10, rsm=0.5, verbose=0, random_seed=42)),
])
pred_cv_finance = np.clip(cross_val_predict(model_finance, texts_train, y_train_finance, cv=kf), 1, 5)
print("MAE Val Финансовая модель", mean_absolute_error(y_train_finance, pred_cv_finance))
model_finance.fit(texts_train, y_train_finance)
pred_test_finance = np.clip(model_finance.predict(texts_test), 1, 5)
print("MAE Test Финансовая модель", mean_absolute_error(y_test_finance, pred_test_finance))

MAE Val Финансовая модель 0.5509522097685038
MAE Test Финансовая модель 0.6201043592180878


In [23]:
#Модель для оценивания критерия Анализ рисков
y_train_risks = df["Анализ рисков"].astype(float).values
y_test_risks = df_test_cases["Анализ рисков"].astype(float).values
model_risks = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True, max_features=20000)),
    ("svd", TruncatedSVD(n_components=100, random_state=42)),
    ("catboost", CatBoostRegressor(iterations=300, depth=3, learning_rate=0.05,
                                   l2_leaf_reg=10, rsm=0.5, verbose=0, random_seed=42)),
])
pred_cv_risks = np.clip(cross_val_predict(model_risks, texts_train, y_train_risks, cv=kf), 1, 5)
print("MAE Val Анализ рисков", mean_absolute_error(y_train_risks, pred_cv_risks))
model_risks.fit(texts_train, y_train_risks)
pred_test_risks = np.clip(model_risks.predict(texts_test), 1, 5)
print("MAE Test Анализ рисков", mean_absolute_error(y_test_risks, pred_test_risks))

MAE Val Анализ рисков 0.5592855269375692
MAE Test Анализ рисков 0.6627069498562705


In [24]:
#Модель для оценивания критерия Доказательства
y_train_proves = df["Доказательства"].astype(float).values
y_test_proves = df_test_cases["Доказательства"].astype(float).values
model_proves = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True, max_features=20000)),
    ("svd", TruncatedSVD(n_components=100, random_state=42)),
    ("catboost", CatBoostRegressor(iterations=300, depth=3, learning_rate=0.05,
                                   l2_leaf_reg=10, rsm=0.5, verbose=0, random_seed=42)),
])
pred_cv_proves = np.clip(cross_val_predict(model_proves, texts_train, y_train_proves, cv=kf), 1, 5)
print("MAE Val Доказательства", mean_absolute_error(y_train_proves, pred_cv_proves))
model_proves.fit(texts_train, y_train_proves)
pred_test_proves = np.clip(model_proves.predict(texts_test), 1, 5)
print("MAE Test Доказательства", mean_absolute_error(y_test_proves, pred_test_proves))

MAE Val Доказательства 0.5815332727236153
MAE Test Доказательства 0.6618582491104302


In [25]:
# Предсказание общей оценки
pred_cv_total = (pred_cv_audience + pred_cv_solution + pred_cv_finance
                 + pred_cv_risks + pred_cv_proves) / 5
pred_test_total = (pred_test_audience + pred_test_solution + pred_test_finance
                   + pred_test_risks + pred_test_proves) / 5

In [26]:
print("Оценка MAE Val", mean_absolute_error(df["Оценка"].astype(float), pred_cv_total))
print("Оценка MAE Test", mean_absolute_error(df_test_cases["Оценка"].astype(float), pred_test_total))

Оценка MAE Val 0.3887901976160385
Оценка MAE Test 0.3746390855476352


In [27]:
import pandas, numpy, sklearn, scipy, catboost, joblib, openpyxl
for m in (pandas, numpy, sklearn, scipy, catboost, joblib, openpyxl):
    print(m.__name__, m.__version__)

pandas 2.2.3
numpy 2.4.6
sklearn 1.6.1
scipy 1.16.3
catboost 1.2.10
joblib 1.6.0
openpyxl 3.1.5
